# PaySprint - FinTech Digital Lending Platform
## Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid", font_scale=1.15)

### 1. Load and Profile Data

In [ ]:
df = pd.read_csv(r'C:\Users\HP\OneDrive\Desktop\Growtech\group4_fintech_paysprint.csv')
print(df.shape)
df.head()

### 2. Data Quality & Cleaning
- Remove duplicates
- Fix categorical variants
- Fix impossible numeric values

In [ ]:
# Drop exact duplicates
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# Clean text columns
text_cols = ['employment_type', 'product_type', 'kyc_status']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# Fix specific variants
df['product_type'] = df['product_type'].replace({'Bnpl': 'BNPL', 'Pl': 'Personal Loan', 'Buy Now Pay Later': 'BNPL', 'Personal loan': 'Personal Loan', 'PERSONAL LOAN': 'Personal Loan'})
df['employment_type'] = df['employment_type'].replace({'Gig-Worker': 'Gig Worker', 'Selfemployed': 'Self Employed', 'Self-Employed': 'Self Employed'})
df['kyc_status'] = df['kyc_status'].replace({'Nan': np.nan, 'None': np.nan})

print("\nCleaned Value Counts for Employment:")
print(df['employment_type'].value_counts())

# Invalid approved amount (exceeds requested)
invalid_amt = df[df['approved_amount_inr'] > df['requested_amount_inr']]
print("\nRows with approved > requested:", len(invalid_amt))
# Fix by capping
df['approved_amount_inr'] = np.where(df['approved_amount_inr'] > df['requested_amount_inr'], df['requested_amount_inr'], df['approved_amount_inr'])

# Invalid ages and credit scores
df.loc[(df['credit_score'] < 300) | (df['credit_score'] > 900), 'credit_score'] = np.nan
df.loc[(df['customer_age'] < 18) | (df['customer_age'] > 80), 'customer_age'] = np.nan
df.loc[df['monthly_income_inr'] < 0, 'monthly_income_inr'] = np.nan

### 3. Blank Values Context
Checking missing values and understanding structurally missing fields.

In [ ]:
print(df[['approved_amount_inr', 'days_past_due', 'rejection_reason']].isnull().sum())
# These are structurally missing. 
# approved_amount_inr is missing because the application was rejected.
# days_past_due is missing because nothing was disbursed.
# rejection_reason is missing because they were approved.

### 4. DTI Ratio Calculation

In [ ]:
df['dti_ratio'] = df['existing_emi_inr'] / df['monthly_income_inr']
df.replace([np.inf, -np.inf], np.nan, inplace=True)
# Cap absurdly high DTI ratios for visualization
df.loc[df['dti_ratio'] > 5, 'dti_ratio'] = np.nan

### 5. Exploratory Data Analysis

In [ ]:
# Rejection Reason Mix
plt.figure(figsize=(10,5))
rej = df[df['application_status'] == 'Rejected']
sns.countplot(y='rejection_reason', data=rej, order=rej['rejection_reason'].value_counts().index, palette='viridis')
plt.title('Distribution of Rejection Reasons')
plt.show()

In [ ]:
# Rejection Rate by Employment Type
df['rejected'] = (df['application_status'] == 'Rejected').astype(int)
rej_by_emp = df.groupby('employment_type')['rejected'].mean().sort_values(ascending=False)
plt.figure(figsize=(8,5))
rej_by_emp.plot(kind='bar', color='#17b8a6')
plt.title('Rejection Rate by Employment Type')
plt.ylabel('Rejection Rate')
plt.show()

In [ ]:
# Default Rate by Employment Type
def_by_emp = df[df['disbursed_flag'] == 1].groupby('employment_type')['default_flag'].mean().sort_values(ascending=False)
plt.figure(figsize=(8,5))
def_by_emp.plot(kind='bar', color='#d84a6b')
plt.title('Default Rate by Employment Type (Disbursed Loans)')
plt.ylabel('Default Rate')
plt.show()

In [ ]:
# Save clean data
df.drop(columns=['rejected'], inplace=True)
df.to_csv('Group4_Cleaned.csv', index=False)
print("Clean data saved to Group4_Cleaned.csv")